# WhatsApp to Excel - Date Range Filter Version

یہ version **date range** کے ساتھ کام کرتا ہے۔

آپ صرف ایک دن کا data لے سکتے ہیں!


## Step 1: Install Packages

In [1]:
import subprocess
import sys

packages = ['selenium', 'openpyxl', 'webdriver-manager', 'pandas']

for package in packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"OK {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"OK {package} installed")

print("\nOK All packages installed!")

OK selenium already installed
OK openpyxl already installed
OK webdriver-manager already installed
OK pandas already installed

OK All packages installed!


## Step 2: Import Libraries

In [2]:
import os
import time
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
import openpyxl
import pandas as pd

print("OK All libraries imported!")

OK All libraries imported!


## Step 3: Define Automation Class with Date Filter

In [3]:
class WhatsAppExcelDateFilter:
    def __init__(self, excel_file_path: str, group_name: str, start_date: str, end_date: str):
        self.excel_file_path = excel_file_path
        self.group_name = group_name
        self.start_date = start_date  # Format: "1-4-2026"
        self.end_date = end_date      # Format: "4-4-2026"
        self.driver = None
        self.processed_messages = set()
        self.load_processed_messages()
        
    def load_processed_messages(self):
        cache_file = Path("processed_messages.json")
        if cache_file.exists():
            with open(cache_file, 'r') as f:
                self.processed_messages = set(json.load(f))
    
    def save_processed_messages(self):
        with open("processed_messages.json", 'w') as f:
            json.dump(list(self.processed_messages), f)
    
    def parse_date(self, date_str: str) -> datetime:
        """Parse date string like '1-4-2026' to datetime"""
        try:
            parts = date_str.split('-')
            day = int(parts[0])
            month = int(parts[1])
            year = int(parts[2])
            return datetime(year, month, day)
        except:
            return None
    
    def is_date_in_range(self, message_date_str: str) -> bool:
        """Check if message date is within the range"""
        msg_date = self.parse_date(message_date_str)
        start = self.parse_date(self.start_date)
        end = self.parse_date(self.end_date)
        
        if not msg_date or not start or not end:
            return False
        
        return start <= msg_date <= end
    
    def setup_driver(self):
        print("\n" + "="*60)
        print("Setting up WhatsApp Web connection...")
        print("="*60)
        
        chrome_options = Options()
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--start-maximized")
        
        user_data_dir = str(Path.home() / ".whatsapp_automation_datefilter")
        chrome_options.add_argument(f"user-data-dir={user_data_dir}")
        
        service = Service(ChromeDriverManager().install())
        self.driver = webdriver.Chrome(service=service, options=chrome_options)
        
        print("Opening WhatsApp Web...")
        self.driver.get("https://web.whatsapp.com")
        
        print("\n" + "!"*60)
        print("SCAN QR CODE WITH YOUR PHONE!")
        print("!"*60 + "\n")
        
        try:
            WebDriverWait(self.driver, 120).until(
                EC.presence_of_all_elements_located((By.XPATH, "//div[@data-testid='chat-list-item']"))
            )
            print("OK Successfully logged in!\n")
            time.sleep(3)
        except Exception as e:
            print(f"ERROR Login failed: {e}\n")
            return False
        
        return True
    
    def find_group(self) -> bool:
        print(f"Looking for group: {self.group_name}...")
        
        try:
            search_box = WebDriverWait(self.driver, 300).until(
                EC.presence_of_element_located((By.XPATH, "//input[@placeholder='Search or start new chat']"))
            )
            search_box.click()
            search_box.clear()
            search_box.send_keys(self.group_name)
            
            time.sleep(2)
            
            group_item = WebDriverWait(self.driver, 10).until(
                EC.presence_of_element_located((By.XPATH, f"//span[contains(text(), '{self.group_name}')]"))
            )
            group_item.click()
            
            print(f"OK Opened group: {self.group_name}\n")
            time.sleep(2)
            return True
            
        except Exception as e:
            print(f"ERROR Could not find group: {e}\n")
            return False
    
    def extract_messages(self) -> List[Dict]:
        messages = []
        
        try:
            message_elements = self.driver.find_elements(By.XPATH, "//div[@data-testid='msg-container']")
            
            for msg_elem in message_elements:
                try:
                    msg_text = msg_elem.text
                    
                    if not msg_text or len(msg_text) < 5:
                        continue
                    
                    msg_id = hash(msg_text) % ((2**31) - 1)
                    
                    if msg_id in self.processed_messages:
                        continue
                    
                    messages.append({
                        'id': msg_id,
                        'text': msg_text,
                        'timestamp': datetime.now().isoformat()
                    })
                    
                    self.processed_messages.add(msg_id)
                    
                except Exception:
                    continue
            
            return messages
            
        except Exception as e:
            print(f"Error extracting messages: {e}")
            return []
    
    def parse_complaint_data(self, message_text: str) -> Optional[Dict]:
        lines = message_text.strip().split('\n')
        if len(lines) < 2:
            return None
        
        data = {}
        
        # Extract time
        time_match = re.search(r'(\d{1,2}):(\d{2})\s*(am|pm|AM|PM)?', lines[0])
        if time_match:
            data['Time'] = time_match.group(0)
        
        # Extract date
        date_match = re.search(r'(\d{1,2})-(\d{1,2})-(\d{4})', message_text)
        if date_match:
            data['Date'] = date_match.group(0)
        else:
            return None  # No date found
        
        # CHECK DATE RANGE
        if not self.is_date_in_range(data['Date']):
            return None  # Date not in range
        
        # Extract branch
        branch_match = re.search(r'([A-Za-z\s]+)\s+Branch', message_text, re.IGNORECASE)
        if branch_match:
            data['Branch'] = branch_match.group(1).strip()
        
        # Extract complaint
        complaint_text = '\n'.join(lines[1:])
        if complaint_text:
            data['Complain'] = complaint_text.strip()[:200]
        
        # Categorize
        complaint_lower = complaint_text.lower()
        if 'staff' in complaint_lower or 'employee' in complaint_lower:
            data['Category'] = 'staff issue'
        elif 'food' in complaint_lower or 'quality' in complaint_lower:
            data['Category'] = 'quality issue'
        elif 'service' in complaint_lower or 'slow' in complaint_lower:
            data['Category'] = 'service issue'
        elif 'waste' in complaint_lower:
            data['Category'] = 'wastage issue'
        else:
            data['Category'] = 'other'
        
        data['Response (Y / N)'] = 'N'
        
        return data if len(data) > 2 else None
    
    def update_excel(self, complaint_data: Dict):
        try:
            if not os.path.exists(self.excel_file_path):
                print(f"ERROR Excel file not found: {self.excel_file_path}")
                return False
            
            wb = openpyxl.load_workbook(self.excel_file_path)
            ws = wb.active
            
            next_row = ws.max_row + 1
            
            columns = {
                'Date': 1,
                'Time': 2,
                'Branch': 3,
                'Category': 4,
                'Complain': 7,
                'Response (Y / N)': 8,
            }
            
            for key, col_num in columns.items():
                if key in complaint_data:
                    ws.cell(row=next_row, column=col_num, value=complaint_data[key])
            
            wb.save(self.excel_file_path)
            print(f"OK Added (Row {next_row}): {complaint_data.get('Complain', 'Unknown')[:40]}...")
            return True
            
        except Exception as e:
            print(f"ERROR updating Excel: {e}")
            return False
    
    def run_once(self):
        """Run once to collect data for the date range"""
        if not self.setup_driver():
            return
        
        if not self.find_group():
            self.driver.quit()
            return
        
        print(f"\nOK Collecting data from {self.start_date} to {self.end_date}...")
        print("Scanning messages...\n")
        
        try:
            for i in range(5):  # Check 5 times
                print(f"Scan #{i+1}...")
                
                messages = self.extract_messages()
                
                if messages:
                    print(f"  Found {len(messages)} new message(s)")
                    
                    for msg in messages:
                        complaint_data = self.parse_complaint_data(msg['text'])
                        
                        if complaint_data:
                            self.update_excel(complaint_data)
                else:
                    print("  No new messages")
                
                self.save_processed_messages()
                time.sleep(5)
        
        except KeyboardInterrupt:
            print("\n\nStopped by user")
        
        finally:
            self.driver.quit()
            print("\nOK Closed WhatsApp Web connection")
            print(f"OK Data collection complete for {self.start_date} to {self.end_date}")

print("OK Automation class defined!")

OK Automation class defined!


## Step 4: Configure Date Range

**یہاں اپنی تاریخ ڈالیں:**

In [4]:
# CONFIGURATION

EXCEL_FILE_PATH = r"C:\Users\GT-Tech\Desktop\Feb_surveillance.xlsm.xlsx"
GROUP_NAME = "Hns Surveillance"

# DATE RANGE - یہاں تاریخ ڈالیں
# Format: "day-month-year" (مثال: "1-4-2026")
START_DATE = "1-4-2026"   # شروع کی تاریخ
END_DATE = "2-4-2026"     # آخری تاریخ (ایک دن کے لیے same رکھیں)

print(f"Configuration:")
print(f"  Excel File: {EXCEL_FILE_PATH}")
print(f"  Group Name: {GROUP_NAME}")
print(f"  Date Range: {START_DATE} to {END_DATE}")
print(f"\nOK Configuration ready!")

Configuration:
  Excel File: C:\Users\GT-Tech\Desktop\Feb_surveillance.xlsm.xlsx
  Group Name: Hns Surveillance
  Date Range: 1-4-2026 to 2-4-2026

OK Configuration ready!


In [10]:
import subprocess
import sys
import time
import os

print("Fixing Chrome driver...")

# Kill Chrome
try:
    os.system("taskkill /F /IM chrome.exe")
    time.sleep(2)
except:
    pass

# Update drivers
subprocess.run(["pip", "uninstall", "-y", "webdriver-manager"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "webdriver-manager"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "selenium"], check=True)

print("OK Fixed! Now run Cell 5 again!")


Fixing Chrome driver...
OK Fixed! Now run Cell 5 again!


## Step 5: Start Data Collection

**یہ cell چلائیں:**

In [5]:
if not os.path.exists(EXCEL_FILE_PATH):
    print(f"ERROR Excel file not found at: {EXCEL_FILE_PATH}")
else:
    print(f"OK Excel file found")
    print(f"OK Group name: {GROUP_NAME}")
    print(f"\n" + "="*60)
    print(f"COLLECTING DATA FROM {START_DATE} TO {END_DATE}")
    print("="*60 + "\n")
    
    automation = WhatsAppExcelDateFilter(EXCEL_FILE_PATH, GROUP_NAME, START_DATE, END_DATE)
    automation.run_once()

OK Excel file found
OK Group name: Hns Surveillance

COLLECTING DATA FROM 1-4-2026 TO 2-4-2026


Setting up WhatsApp Web connection...
Opening WhatsApp Web...

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
SCAN QR CODE WITH YOUR PHONE!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

ERROR Login failed: Message: 




## اگلے دن کے لیے:

**Step 4 میں تاریخ بدلیں:**

```python
START_DATE = "2-4-2026"   # اگلی تاریخ
END_DATE = "2-4-2026"
```

پھر Step 5 دوبارہ چلائیں۔ نیا data Excel میں شامل ہو جائے گا! ✅